# Parcel ↔ Image Matching (cleaned final notebook)

This notebook rebuilds the pipeline from scratch with cleaner logic and a few accuracy improvements:

1. Extract image GPS + heading from EXIF  
2. Build projected camera points and triangular view cones  
3. Load parcel polygons in a projected CRS  
4. Generate candidate matches via spatial join  
5. Score candidates using **boundary distance + heading alignment**  
6. Select **one best parcel per image**  
7. Aggregate matched images **by parcel** for export and Folium visualization  

### Important logic choices
- **Best match** is selected **per image**
- **Folium markers** are plotted **per parcel**, so marker count can be smaller than image count
- Distance is computed to the **parcel boundary/polygon**, not only to the centroid
- Heading is used in the final score, not only in cone generation

In [2]:
from pathlib import Path
import os
import math
from html import escape

import numpy as np
import pandas as pd
import geopandas as gpd
import folium

from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS
from shapely.geometry import Point, Polygon
from shapely.ops import nearest_points

# -----------------------------
# Paths and core parameters
# -----------------------------
IMAGE_FOLDER = Path("Data/BBxBImages")  # Original image folder
PARCEL_FILE  = Path("Data/Real_Property_Information.geojson")  # Baltimore parcel dataset

CRS_WGS84 = "EPSG:4326"
CRS_PROJ  = "EPSG:2248"

FOV_DEG = 50
CONE_LENGTH_FT = 130
MAX_BOUNDARY_DISTANCE_FT = 150
ANGLE_WEIGHT_FT_PER_DEG = 1.0
output_dir = Path("outputs")
IMAGE_METADATA_CSV = Path("image_metadata_extracted.csv")
BEST_MATCH_CSV = Path("image_best_match.csv")
PARCEL_MATCH_CSV = Path("parcel_image_join.csv")
PARCEL_MATCH_GEOJSON = Path("parcel_image_join.geojson")
FOLIUM_HTML = Path("parcel_map.html")

print("Image folder exists:", IMAGE_FOLDER.exists(), IMAGE_FOLDER)
print("Parcel file exists :", PARCEL_FILE.exists(), PARCEL_FILE)

Image folder exists: True Data/BBxBImages
Parcel file exists : True Data/Real_Property_Information.geojson


In [3]:
def get_exif_data(image_path):
    try:
        img = Image.open(image_path)
        exif = img._getexif()
        if not exif:
            return {}
        return {TAGS.get(tag, tag): value for tag, value in exif.items()}
    except Exception:
        return {}

def get_gps_info(exif_data):
    gps = exif_data.get("GPSInfo")
    if not gps:
        return None
    return {GPSTAGS.get(k, k): v for k, v in gps.items()}

def _rational_to_float(x):
    try:
        if hasattr(x, "numerator") and hasattr(x, "denominator"):
            return float(x.numerator) / float(x.denominator)
        if isinstance(x, tuple) and len(x) == 2 and all(isinstance(v, (int, float)) for v in x):
            num, den = x
            return float(num) / float(den)
        return float(x)
    except Exception:
        return float("nan")

def dms_to_decimal(dms):
    try:
        d, m, s = dms
        d = _rational_to_float(d)
        m = _rational_to_float(m)
        s = _rational_to_float(s)
        return d + m / 60.0 + s / 3600.0
    except Exception:
        return None

def extract_lat_lon_heading(image_path):
    exif_data = get_exif_data(image_path)
    gps_info = get_gps_info(exif_data)
    if not gps_info:
        return None, None, None

    lat = dms_to_decimal(gps_info.get("GPSLatitude"))
    lon = dms_to_decimal(gps_info.get("GPSLongitude"))
    if lat is None or lon is None:
        return None, None, None

    if gps_info.get("GPSLatitudeRef", "N") != "N":
        lat = -lat
    if gps_info.get("GPSLongitudeRef", "E") != "E":
        lon = -lon

    heading = gps_info.get("GPSImgDirection", None)
    try:
        heading = _rational_to_float(heading) if heading is not None else None
        if pd.notna(heading):
            heading = float(heading) % 360
        else:
            heading = None
    except Exception:
        heading = None

    return lat, lon, heading

In [4]:
rows = []

for fname in sorted(os.listdir(IMAGE_FOLDER)):
    if fname.lower().endswith(".jpg"):
        path = IMAGE_FOLDER / fname
        lat, lon, heading = extract_lat_lon_heading(path)
        rows.append(
            {
                "filename": fname,
                "image_path": str(path.resolve()),
                "latitude": lat,
                "longitude": lon,
                "heading": heading,
            }
        )

df_imgs = pd.DataFrame(rows)
df_imgs.to_csv(IMAGE_METADATA_CSV, index=False)

print("Raw image rows:", len(df_imgs))
print("With lat/lon   :", df_imgs[["latitude", "longitude"]].dropna().shape[0])
print("With heading   :", df_imgs["heading"].notna().sum())
display(df_imgs.head())

Raw image rows: 578
With lat/lon   : 578
With heading   : 578


,filename,image_path,latitude,longitude,heading
0,Run 10_Camera 4 360_115_1.jpg,/Users/jiaweihu/Desktop/BBxB/Spatial Join/Data...,39.303667,-76.583810,175.531865
1,Run 10_Camera 4 360_130_3.jpg,/Users/jiaweihu/Desktop/BBxB/Spatial Join/Data...,39.294092,-76.654371,356.000542
2,Run 10_Camera 4 360_136_1.jpg,/Users/jiaweihu/Desktop/BBxB/Spatial Join/Data...,39.294100,-76.654161,357.220324
3,Run 10_Camera 4 360_137_1.jpg,/Users/jiaweihu/Desktop/BBxB/Spatial Join/Data...,39.294101,-76.654126,357.246696
4,Run 10_Camera 4 360_141_1.jpg,/Users/jiaweihu/Desktop/BBxB/Spatial Join/Data...,39.294107,-76.653986,352.254918


## Geometry model

The camera field of view is approximated as a **triangular view cone** in a projected CRS.

That is a simplification, but it is:
- easy to explain,
- fast to compute,
- usually good enough for candidate generation.

In [5]:
def heading_to_math_angle(heading_deg):
    return (90 - float(heading_deg)) % 360

def point_from_xy_angle(x, y, angle_deg, dist):
    ang = math.radians(angle_deg)
    return (x + dist * math.cos(ang), y + dist * math.sin(ang))

def create_view_cone_projected(x, y, heading_deg, cone_length=CONE_LENGTH_FT, fov_deg=FOV_DEG):
    center_ang = heading_to_math_angle(heading_deg)
    half = fov_deg / 2.0
    left_pt  = point_from_xy_angle(x, y, center_ang + half, cone_length)
    right_pt = point_from_xy_angle(x, y, center_ang - half, cone_length)
    return Polygon([(x, y), left_pt, right_pt])

def bearing_from_points_projected(p1, p2):
    dx = p2.x - p1.x
    dy = p2.y - p1.y
    ang_math = math.degrees(math.atan2(dy, dx)) % 360
    return (90 - ang_math) % 360

def smallest_angle_diff_deg(a, b):
    diff = abs(float(a) - float(b)) % 360
    return min(diff, 360 - diff)

In [6]:
df_valid = df_imgs.dropna(subset=["latitude", "longitude", "heading"]).copy()

gdf_images = gpd.GeoDataFrame(
    df_valid,
    geometry=gpd.points_from_xy(df_valid["longitude"], df_valid["latitude"]),
    crs=CRS_WGS84,
).to_crs(CRS_PROJ)

gdf_images["camera_point"] = gdf_images.geometry.copy()
gdf_images["geometry"] = gdf_images.apply(
    lambda r: create_view_cone_projected(
        r["camera_point"].x,
        r["camera_point"].y,
        r["heading"],
        cone_length=CONE_LENGTH_FT,
        fov_deg=FOV_DEG,
    ),
    axis=1,
)

print("Image rows after valid GPS+heading filter:", len(gdf_images))
display(gdf_images[["filename", "latitude", "longitude", "heading"]].head())

Image rows after valid GPS+heading filter: 578


,filename,latitude,longitude,heading
0,Run 10_Camera 4 360_115_1.jpg,39.303667,-76.583810,175.531865
1,Run 10_Camera 4 360_130_3.jpg,39.294092,-76.654371,356.000542
2,Run 10_Camera 4 360_136_1.jpg,39.294100,-76.654161,357.220324
3,Run 10_Camera 4 360_137_1.jpg,39.294101,-76.654126,357.246696
4,Run 10_Camera 4 360_141_1.jpg,39.294107,-76.653986,352.254918


In [7]:
gdf_parcels = gpd.read_file(PARCEL_FILE)

if "BLOCKLOT" in gdf_parcels.columns and "property_id" not in gdf_parcels.columns:
    gdf_parcels = gdf_parcels.rename(columns={"BLOCKLOT": "property_id"})

gdf_parcels = gdf_parcels.to_crs(CRS_PROJ).copy()
gdf_parcels["parcel_geom"] = gdf_parcels.geometry
gdf_parcels["parcel_centroid_proj"] = gdf_parcels.geometry.centroid
gdf_parcels["parcel_rep_pt_proj"] = gdf_parcels.geometry.representative_point()

parcel_centroids_wgs84 = gpd.GeoSeries(
    gdf_parcels["parcel_centroid_proj"], crs=CRS_PROJ
).to_crs(CRS_WGS84)
gdf_parcels["parcel_latitude"] = parcel_centroids_wgs84.y
gdf_parcels["parcel_longitude"] = parcel_centroids_wgs84.x

# ---------------------------------------------------------
# Clean / standardize new Baltimore property-level fields
# Raw source columns in this GeoJSON:
#   SALEPRIC -> Property Sale Price
#   LDATE    -> Last Update (MMDDYYYY)
#   VACIND   -> Vacant Indicator (Y / N / missing)
# ---------------------------------------------------------

def parse_mmddyyyy_series(series):
    return pd.to_datetime(series.astype(str).str.strip(), format="%m%d%Y", errors="coerce")

if "SALEPRIC" in gdf_parcels.columns:
    gdf_parcels["property_sale_price"] = pd.to_numeric(gdf_parcels["SALEPRIC"], errors="coerce")
else:
    gdf_parcels["property_sale_price"] = np.nan

if "LDATE" in gdf_parcels.columns:
    gdf_parcels["last_update"] = parse_mmddyyyy_series(gdf_parcels["LDATE"])
    gdf_parcels["last_update_str"] = gdf_parcels["last_update"].dt.strftime("%Y-%m-%d")
else:
    gdf_parcels["last_update"] = pd.NaT
    gdf_parcels["last_update_str"] = pd.NA

if "VACIND" in gdf_parcels.columns:
    gdf_parcels["vacant_indicator"] = (
        gdf_parcels["VACIND"]
        .map({"Y": 1, "N": 0})
        .astype(float)
    )
else:
    gdf_parcels["vacant_indicator"] = np.nan

print("Parcel polygons loaded:", len(gdf_parcels))
display(
    gdf_parcels[
        [c for c in [
            "property_id", "FULLADDR", "parcel_latitude", "parcel_longitude",
            "SALEPRIC", "property_sale_price", "LDATE", "last_update_str",
            "VACIND", "vacant_indicator"
        ] if c in gdf_parcels.columns]
    ].head()
)

Parcel polygons loaded: 238491


,property_id,FULLADDR,parcel_latitude,parcel_longitude,SALEPRIC,property_sale_price,LDATE,last_update_str,VACIND,vacant_indicator
0,0001 001,2045 W NORTH AVE,39.309421,-76.651146,250000,250000,03082026,2026-03-08,None,NaN
1,0001 002,2043 W NORTH AVE,39.309424,-76.651093,250000,250000,03082026,2026-03-08,None,NaN
2,0001 003,2041 W NORTH AVE,39.309426,-76.651043,0,0,03082026,2026-03-08,Y,1.0
3,0001 004,2039 W NORTH AVE,39.309427,-76.650992,0,0,03082026,2026-03-08,Y,1.0
4,0001 005,2037 W NORTH AVE,39.309429,-76.650944,70000,70000,03082026,2026-03-08,None,NaN


In [8]:
candidate_cols_left = ["filename", "image_path", "latitude", "longitude", "heading", "camera_point", "geometry"]
candidate_cols_right = ["property_id", "FULLADDR", "geometry", "parcel_geom", "parcel_centroid_proj", "parcel_rep_pt_proj"]

candidates_all = gpd.sjoin(
    gdf_images[candidate_cols_left],
    gdf_parcels[candidate_cols_right],
    how="left",
    predicate="intersects",
).copy()

def compute_candidate_metrics(row):
    if pd.isna(row["property_id"]):
        return pd.Series(
            {
                "distance_boundary_ft": None,
                "distance_centroid_ft": None,
                "bearing_to_parcel_deg": None,
                "angle_diff_deg": None,
                "match_score": None,
            }
        )

    camera_pt = row["camera_point"]
    parcel_geom = row["parcel_geom"]
    parcel_centroid = row["parcel_centroid_proj"]
    parcel_rep_pt = row["parcel_rep_pt_proj"]

    distance_boundary = camera_pt.distance(parcel_geom)
    distance_centroid = camera_pt.distance(parcel_centroid)
    bearing = bearing_from_points_projected(camera_pt, parcel_rep_pt)
    angle_diff = smallest_angle_diff_deg(row["heading"], bearing)
    score = distance_boundary + ANGLE_WEIGHT_FT_PER_DEG * angle_diff

    return pd.Series(
        {
            "distance_boundary_ft": distance_boundary,
            "distance_centroid_ft": distance_centroid,
            "bearing_to_parcel_deg": bearing,
            "angle_diff_deg": angle_diff,
            "match_score": score,
        }
    )

candidates_all = pd.concat([candidates_all, candidates_all.apply(compute_candidate_metrics, axis=1)], axis=1)

print("Candidate rows:", len(candidates_all))
display(
    candidates_all[
        ["filename", "property_id", "FULLADDR", "distance_boundary_ft", "distance_centroid_ft", "angle_diff_deg", "match_score"]
    ].head(10)
)

Candidate rows: 3920


,filename,property_id,FULLADDR,distance_boundary_ft,distance_centroid_ft,angle_diff_deg,match_score
0,Run 10_Camera 4 360_115_1.jpg,1572 036,2333 E CHASE ST,101.826092,254.653143,6.972253,108.798345
0,Run 10_Camera 4 360_115_1.jpg,1572 001,2401 E CHASE ST,38.944792,74.358551,15.031039,53.975832
0,Run 10_Camera 4 360_115_1.jpg,1572 002,2403 E CHASE ST,37.500657,72.632345,4.771832,42.272489
0,Run 10_Camera 4 360_115_1.jpg,1572 003,2405 E CHASE ST,37.583335,72.972744,5.004073,42.587408
0,Run 10_Camera 4 360_115_1.jpg,1572 004,2407 E CHASE ST,40.273320,75.380865,14.162544,54.435864
0,Run 10_Camera 4 360_115_1.jpg,1572 005,2409 E CHASE ST,46.278235,79.703421,23.136368,69.414603
0,Run 10_Camera 4 360_115_1.jpg,1572 006,2411 E CHASE ST,54.556064,85.395816,30.500463,85.056528
0,Run 10_Camera 4 360_115_1.jpg,1572 007,2413 E CHASE ST,63.445617,92.501335,36.410574,99.856191
1,Run 10_Camera 4 360_130_3.jpg,2157 001,400 N SMALLWOOD ST,0.000000,183.839864,92.316274,92.316274
1,Run 10_Camera 4 360_130_3.jpg,2201 044,2340 LAURETTA AVE,32.831915,70.219628,15.832386,48.664301


In [9]:
candidates = candidates_all[
    candidates_all["property_id"].notna()
    & candidates_all["distance_boundary_ft"].notna()
    & (candidates_all["distance_boundary_ft"] <= MAX_BOUNDARY_DISTANCE_FT)
].copy()

print("Images total                     :", len(df_imgs))
print("Images with valid GPS+heading    :", gdf_images["filename"].nunique())
print("Images with >=1 parcel candidate :", candidates_all.loc[candidates_all["property_id"].notna(), "filename"].nunique())
print("Images after distance filter     :", candidates["filename"].nunique())
print()
print("Boundary distance stats:")
display(candidates["distance_boundary_ft"].describe())
print("Angle diff stats:")
display(candidates["angle_diff_deg"].describe())

Images total                     : 578
Images with valid GPS+heading    : 578
Images with >=1 parcel candidate : 578
Images after distance filter     : 578

Boundary distance stats:


count    3920.000000
mean       65.912762
std        24.674249
min         0.000000
25%        47.783987
50%        62.750163
75%        76.528435
max       129.172203
Name: distance_boundary_ft, dtype: float64

Angle diff stats:


count    3920.000000
mean       19.807169
std        13.126804
min         0.003565
25%         9.055564
50%        18.806442
75%        28.254810
max        98.698394
Name: angle_diff_deg, dtype: float64

In [10]:
# --- 12. Select one best parcel per image ---

# 如果你前面已经把 candidates 定义为过滤后的候选，就继续用它。
# 如果没有，这里可以明确写成 candidates_valid = candidates.dropna(subset=["property_id"]).copy()
candidates_valid = candidates.dropna(subset=["property_id"]).copy()

best_match = (
    candidates_valid
    .sort_values(["filename", "match_score", "distance_boundary_ft", "angle_diff_deg"])
    .groupby("filename", as_index=False)
    .first()
    .copy()
)

# 重新显式设回 GeoDataFrame，避免 groupby 后 CRS 丢失
best_match = gpd.GeoDataFrame(best_match, geometry="geometry", crs=gdf_images.crs)

print("Best-matched images:", len(best_match))
print("Unique matched parcels:", best_match["property_id"].nunique())
display(
    best_match[
        ["filename", "property_id", "FULLADDR", "distance_boundary_ft", "angle_diff_deg", "match_score"]
    ].head(15)
)

Best-matched images: 578
Unique matched parcels: 502


,filename,property_id,FULLADDR,distance_boundary_ft,angle_diff_deg,match_score
0,Run 10_Camera 4 360_115_1.jpg,1572 002,2403 E CHASE ST,37.500657,4.771832,42.272489
1,Run 10_Camera 4 360_130_3.jpg,2201 045,2342 LAURETTA AVE,30.966650,4.655507,35.622157
2,Run 10_Camera 4 360_136_1.jpg,2201 041,2334 LAURETTA AVE,30.815083,1.062929,31.878012
3,Run 10_Camera 4 360_137_1.jpg,2201 040,2332 LAURETTA AVE,31.042462,2.181825,33.224287
4,Run 10_Camera 4 360_141_1.jpg,2201 038,2328 LAURETTA AVE,30.573625,3.411100,33.984725
5,Run 10_Camera 4 360_14_1.jpg,3801 014,1927 SAINT PAUL ST,34.090839,2.084824,36.175663
6,Run 10_Camera 4 360_151_1.jpg,2201 030,2312 LAURETTA AVE,30.996213,1.133748,32.129961
7,Run 10_Camera 4 360_152_1.jpg,2201 029,2310 LAURETTA AVE,31.323992,2.028102,33.352094
8,Run 10_Camera 4 360_169_1.jpg,2157 001,400 N SMALLWOOD ST,0.000000,20.572176,20.572176
9,Run 10_Camera 4 360_192_3.jpg,1572 037,2301 E CHASE ST,43.933759,49.378880,93.312638


In [11]:
# --- 13. Aggregate matched images to parcel level ---

parcel_image_agg = (
    best_match.groupby("property_id", as_index=False)
    .agg(
        image_names=("filename", lambda s: "|".join(sorted(set(map(str, s))))),
        image_paths=("image_path", lambda s: "|".join(sorted(set(map(str, s))))),
        image_count=("filename", "nunique"),
        matched_image_latitude_mean=("latitude", "mean"),
        matched_image_longitude_mean=("longitude", "mean"),
        min_boundary_distance_ft=("distance_boundary_ft", "min"),
        mean_boundary_distance_ft=("distance_boundary_ft", "mean"),
        min_angle_diff_deg=("angle_diff_deg", "min"),
        mean_angle_diff_deg=("angle_diff_deg", "mean"),
    )
)

# 这是“全量 parcel 表 + 匹配信息”
parcel_all = gdf_parcels.merge(parcel_image_agg, on="property_id", how="left").copy()
parcel_all = gpd.GeoDataFrame(parcel_all, geometry="geometry", crs=gdf_parcels.crs)

# 这才是真正“成功匹配到至少一张图片”的 parcel
parcel_matches = parcel_all.dropna(subset=["image_count"]).copy()
parcel_matches = gpd.GeoDataFrame(parcel_matches, geometry="geometry", crs=gdf_parcels.crs)

print("Unique parcels matched:", len(parcel_matches))
display(
    parcel_matches[
        ["property_id", "FULLADDR", "image_count", "image_names",
         "min_boundary_distance_ft", "mean_angle_diff_deg"]
    ].head(10)
)

Unique parcels matched: 505


,property_id,FULLADDR,image_count,image_names,min_boundary_distance_ft,mean_angle_diff_deg
3,0001 004,2039 W NORTH AVE,1.0,Run 18_Camera 4 360_276_3.jpg,56.806587,3.455000
8,0001 009,2029 W NORTH AVE,1.0,Run 18_Camera 4 360_268_1.jpg,56.979923,3.848565
9,0001 010,2027 W NORTH AVE,1.0,Run 18_Camera 4 360_267_1.jpg,57.064350,1.441362
15,0001 016,2001 W NORTH AVE,1.0,Run 18_Camera 4 360_256_3.jpg,57.889231,13.930158
68,0002 015,1901 W NORTH AVE,1.0,Run 18_Camera 4 360_199_1.jpg,55.365776,5.231363
71,0002 018,1907 W NORTH AVE,1.0,Run 18_Camera 4 360_204_3.jpg,56.142530,1.056664
80,0002 028,1929 W NORTH AVE,1.0,Run 18_Camera 4 360_223_3.jpg,57.484881,5.033667
84,0002 032,1937 W NORTH AVE,1.0,Run 18_Camera 4 360_230_3.jpg,57.461936,3.332913
134,0003 001,1800 N FULTON AVE,2.0,Run 23_Camera 4 360_140_1.jpg|Run 23_Camera 4 ...,72.793075,3.041846
135,0003 002,1802 N FULTON AVE,1.0,Run 23_Camera 4 360_143_3.jpg,72.821862,1.441202


## Manual feature labels → parcel-level dataframe

This section merges the uploaded manual annotation CSV onto the image-level matches, then aggregates those labels to the parcel level so `df_parcel` contains:

- unique property ID
- parcel centroid latitude / longitude
- associated image names and file paths
- one column per manually annotated feature
- optional counts showing how many matched images expressed each feature

Aggregation rule used below:
- binary parcel feature = **1 if any matched image for that parcel has the feature**
- feature count = number of matched matched images with that feature


In [12]:

# --- 14. Merge manual image labels and build feature-augmented df_parcel ---

LABEL_FILE = Path("Data/Baltimore_StreetViewImages_Labels.csv")  # adjust if needed

label_df = pd.read_csv(LABEL_FILE).copy()
label_df["filename"] = label_df["Filename"].astype(str).str.strip()

feature_cols = [
    "Crooked_Roofline",
    "Broken_Steps",
    "Missing_Bricks",
    "Peeling_Paint",
    "Boarded_Windows",
    "Boarded_Doors",
    "Plant_Overgrowth",
    "Fire_Damage",
    "Cracked_Sidewalk",
    "Green_Space",
]

# Coerce labels to numeric 0/1
for col in feature_cols:
    label_df[col] = pd.to_numeric(label_df[col], errors="coerce").fillna(0).astype(int)

# Attach labels to image-level best matches
best_match_labeled = best_match.merge(
    label_df[["filename"] + feature_cols],
    on="filename",
    how="left",
)

for col in feature_cols:
    best_match_labeled[col] = best_match_labeled[col].fillna(0).astype(int)

print("Image-level labeled matches:", len(best_match_labeled))
print("Matched images with at least one positive label:",
      int((best_match_labeled[feature_cols].sum(axis=1) > 0).sum()))
display(best_match_labeled[["filename", "property_id"] + feature_cols].head())


Image-level labeled matches: 578
Matched images with at least one positive label: 235


,filename,property_id,Crooked_Roofline,Broken_Steps,Missing_Bricks,Peeling_Paint,Boarded_Windows,Boarded_Doors,Plant_Overgrowth,Fire_Damage,Cracked_Sidewalk,Green_Space
0,Run 10_Camera 4 360_115_1.jpg,1572 002,0,1,0,1,1,1,1,0,0,0
1,Run 10_Camera 4 360_130_3.jpg,2201 045,0,0,0,1,0,0,0,0,0,0
2,Run 10_Camera 4 360_136_1.jpg,2201 041,0,0,0,1,0,0,1,0,0,1
3,Run 10_Camera 4 360_137_1.jpg,2201 040,0,0,0,0,0,0,0,0,0,1
4,Run 10_Camera 4 360_141_1.jpg,2201 038,0,0,0,1,1,1,1,0,0,0


In [13]:
# Aggregate image labels to parcel level
parcel_feature_any = (
    best_match_labeled.groupby("property_id", as_index=False)[feature_cols]
    .max()
)

parcel_feature_counts = (
    best_match_labeled.groupby("property_id", as_index=False)[feature_cols]
    .sum()
    .rename(columns={c: f"{c}_image_count" for c in feature_cols})
)

df_parcel = (
    parcel_matches
    .merge(parcel_feature_any, on="property_id", how="left")
    .merge(parcel_feature_counts, on="property_id", how="left")
    .copy()
)

for col in feature_cols:
    df_parcel[col] = df_parcel[col].fillna(0).astype(int)
    df_parcel[f"{col}_image_count"] = df_parcel[f"{col}_image_count"].fillna(0).astype(int)

# Keep geometry / CRS explicit
df_parcel = gpd.GeoDataFrame(df_parcel, geometry="geometry", crs=parcel_matches.crs)

print("df_parcel rows:", len(df_parcel))
print("Unique properties:", df_parcel["property_id"].nunique())
display(
    df_parcel[
        [c for c in [
            "property_id", "FULLADDR", "parcel_latitude", "parcel_longitude", "image_count",
            "property_sale_price", "last_update_str", "vacant_indicator"
        ] + feature_cols if c in df_parcel.columns]
    ].head(10)
)

df_parcel rows: 505
Unique properties: 502


,property_id,FULLADDR,parcel_latitude,parcel_longitude,image_count,property_sale_price,last_update_str,vacant_indicator,Crooked_Roofline,Broken_Steps,Missing_Bricks,Peeling_Paint,Boarded_Windows,Boarded_Doors,Plant_Overgrowth,Fire_Damage,Cracked_Sidewalk,Green_Space
0,0001 004,2039 W NORTH AVE,39.309427,-76.650992,1.0,0,2026-03-08,1.0,0,0,0,0,0,0,0,0,0,0
1,0001 009,2029 W NORTH AVE,39.309435,-76.650748,1.0,0,2026-03-08,NaN,0,0,0,0,0,0,0,0,0,0
2,0001 010,2027 W NORTH AVE,39.309437,-76.650698,1.0,60000,2026-03-08,0.0,0,0,0,0,0,0,0,0,0,0
3,0001 016,2001 W NORTH AVE,39.309453,-76.650207,1.0,0,2026-03-08,NaN,0,0,0,0,0,0,0,0,0,0
4,0002 015,1901 W NORTH AVE,39.309544,-76.648289,1.0,43500,2026-03-08,1.0,0,0,0,0,0,0,0,0,0,0
5,0002 018,1907 W NORTH AVE,39.309536,-76.648440,1.0,0,2026-03-08,1.0,0,0,0,0,0,0,0,0,0,0
6,0002 028,1929 W NORTH AVE,39.309503,-76.649142,1.0,150000,2026-03-08,0.0,0,0,0,0,0,0,0,0,0,0
7,0002 032,1937 W NORTH AVE,39.309491,-76.649340,1.0,3000,2026-03-08,1.0,0,0,0,0,0,0,0,0,0,0
8,0003 001,1800 N FULTON AVE,39.308955,-76.646714,2.0,0,2026-03-08,1.0,0,0,0,0,0,0,0,0,0,0
9,0003 002,1802 N FULTON AVE,39.308997,-76.646719,1.0,20900,2026-03-08,NaN,0,0,0,0,0,0,0,0,0,0


In [14]:
# --- 15. Export feature-augmented parcel outputs (CSV / GeoJSON / Shapefile) ---

from pathlib import Path

df_parcel_export = df_parcel.copy()

# ------------------------------------------------------------------
# 1) Preserve projected helper geometries as WKT for flat-file export
# ------------------------------------------------------------------
extra_geom_cols_parcel = ["parcel_geom", "parcel_centroid_proj", "parcel_rep_pt_proj"]
for col in extra_geom_cols_parcel:
    if col in df_parcel_export.columns:
        df_parcel_export[f"{col}_wkt"] = df_parcel_export[col].to_wkt()

# Create output folder
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

csv_path = output_dir / "df_parcel_feature_augmented.csv"
geojson_path = output_dir / "df_parcel_feature_augmented.geojson"
shp_base = "df_parcel_feature_augmented"
shp_path = output_dir / f"{shp_base}.shp"

# ------------------------------------------------------------------
# 2) GeoJSON export (WGS84)
# ------------------------------------------------------------------
df_parcel_geojson = df_parcel_export.drop(
    columns=[c for c in extra_geom_cols_parcel if c in df_parcel_export.columns],
    errors="ignore"
).copy()

df_parcel_geojson = gpd.GeoDataFrame(
    df_parcel_geojson,
    geometry="geometry",
    crs=df_parcel.crs
).to_crs(CRS_WGS84)

# Remove old GeoJSON if it exists
if geojson_path.exists():
    geojson_path.unlink()

df_parcel_geojson.to_file(geojson_path, driver="GeoJSON")

# ------------------------------------------------------------------
# 3) Shapefile export
#    - shorten field names
#    - keep only shapefile-safe columns
#    - remove old sidecar files before writing
# ------------------------------------------------------------------
df_parcel_shp = df_parcel_geojson.copy().rename(columns={
    "property_sale_price": "sale_price",
    "last_update_str": "last_upd",
    "vacant_indicator": "vacant_ind",
})

shp_keep_cols = [
    "property_id",
    "FULLADDR",
    "image_count",
    "sale_price",
    "last_upd",
    "vacant_ind",
    "parcel_latitude",
    "parcel_longitude",
    "geometry",
]

df_parcel_shp = df_parcel_shp[
    [c for c in shp_keep_cols if c in df_parcel_shp.columns]
].copy()

# Remove old shapefile sidecar files if they exist
for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg"]:
    f = output_dir / f"{shp_base}{ext}"
    if f.exists():
        try:
            f.unlink()
        except PermissionError:
            print(f"Warning: could not delete {f}; it may be open in another program.")

df_parcel_shp.to_file(shp_path, driver="ESRI Shapefile")

# ------------------------------------------------------------------
# 4) CSV export
# ------------------------------------------------------------------
df_parcel_csv = df_parcel_export.copy()
df_parcel_csv["geometry_wkt"] = df_parcel_csv.geometry.to_wkt()
df_parcel_csv = pd.DataFrame(df_parcel_csv.drop(columns="geometry"))

if csv_path.exists():
    csv_path.unlink()

df_parcel_csv.to_csv(csv_path, index=False)

# ------------------------------------------------------------------
# 5) Print saved paths
# ------------------------------------------------------------------
print(f"Saved -> {csv_path.resolve()}")
print(f"Saved -> {geojson_path.resolve()}")
print(f"Saved -> {shp_path.resolve()}")

Saved -> /Users/jiaweihu/Desktop/BBxB/Spatial Join/outputs/df_parcel_feature_augmented.csv
Saved -> /Users/jiaweihu/Desktop/BBxB/Spatial Join/outputs/df_parcel_feature_augmented.geojson
Saved -> /Users/jiaweihu/Desktop/BBxB/Spatial Join/outputs/df_parcel_feature_augmented.shp


/var/folders/y_/0hljknn13vb1y77zbd8j0lq80000gn/T/ipykernel_32381/954806155.py:81: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  df_parcel_shp.to_file(shp_path, driver="ESRI Shapefile")
/Users/jiaweihu/miniconda3/envs/jupyterlab/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'property_id' to 'property_i'
  ogr_write(
/Users/jiaweihu/miniconda3/envs/jupyterlab/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'image_count' to 'image_coun'
  ogr_write(
/Users/jiaweihu/miniconda3/envs/jupyterlab/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'parcel_latitude' to 'parcel_lat'
  ogr_write(
/Users/jiaweihu/miniconda3/envs/jupyterlab/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'parcel_longitude' to 'parcel_lon'
  ogr_write(


In [27]:
# --- 16. Single Folium map with true three-filter UI:
#         (1) Status filter
#         (2) Building condition checkbox filter
#         (3) Optional sale price range filter
#         + optional sale-price color mode
#         + local image preview in popups

import json
from pathlib import Path
from html import escape
from branca.element import Element

df_parcel_wgs84 = df_parcel.to_crs(CRS_WGS84).copy()

# ------------------------------------------------------------------
# 1) Map setup
# ------------------------------------------------------------------
m_features = folium.Map(
    location=[
        df_parcel_wgs84["parcel_latitude"].mean(),
        df_parcel_wgs84["parcel_longitude"].mean()
    ],
    zoom_start=16,
    tiles="CartoDB positron",
)

MAX_IMAGES_PER_PROPERTY = 2  # show only first 2 images in popup


# ------------------------------------------------------------------
# 2) Helper functions
# ------------------------------------------------------------------
def path_to_file_uri(path_str):
    """Convert local file path to browser-readable file URI."""
    try:
        path_obj = Path(path_str).expanduser().resolve()
        if not path_obj.exists():
            return None
        return path_obj.as_uri()
    except Exception:
        return None


def format_sale_price(value):
    if pd.isna(value):
        return "NA"
    return f"${value:,.0f}"


def format_vacant(value):
    if pd.isna(value):
        return "NA"
    return str(int(value))


def prettify_feature_name(name):
    mapping = {
        "Crooked_Roofline": "Crooked Roofline",
        "Broken_Steps": "Broken Steps",
        "Missing_Bricks": "Missing Bricks",
        "Peeling_Paint": "Peeling Paint",
        "Boarded_Windows": "Boarded Windows",
        "Boarded_Doors": "Boarded Doors",
        "Plant_Overgrowth": "Plant Overgrowth",
        "Fire_Damage": "Fire Damage",
        "Cracked_Sidewalk": "Cracked Sidewalk",
        "Green_Space": "Green Space",
    }
    return mapping.get(name, name.replace("_", " "))


def make_property_popup(row):
    image_paths = str(row.get("image_paths", "")).split("|") if pd.notna(row.get("image_paths")) else []
    image_names = str(row.get("image_names", "")).split("|") if pd.notna(row.get("image_names")) else []

    positive_features = [
        prettify_feature_name(c)
        for c in feature_cols
        if pd.notna(row.get(c)) and int(row.get(c, 0)) == 1
    ]
    feature_html = ", ".join(positive_features) if positive_features else "None"

    sale_price_str = format_sale_price(row.get("property_sale_price", np.nan))
    last_update_str = row.get("last_update_str", "NA")
    vacant_str = format_vacant(row.get("vacant_indicator", np.nan))

    image_blocks = []
    for name, path in list(zip(image_names, image_paths))[:MAX_IMAGES_PER_PROPERTY]:
        file_uri = path_to_file_uri(path)

        image_html = (
            f'<img src="{file_uri}" style="width:260px;max-width:100%;border-radius:6px;border:1px solid #ccc;">'
            if file_uri else
            "<div style='color:#b00;'>Image file not found locally</div>"
        )

        image_blocks.append(
            f"""
            <div style="margin-bottom:12px;padding-bottom:10px;border-bottom:1px solid #eee;">
                <div style="font-size:12px;font-weight:600;margin-bottom:4px;">{escape(str(name))}</div>
                <div style="font-size:11px;color:#666;word-break:break-all;margin-bottom:6px;">{escape(str(path))}</div>
                {image_html}
            </div>
            """
        )

    popup_html = f"""
    <div style="font-family:sans-serif;width:340px;max-height:430px;overflow-y:auto;">
      <div style="font-size:14px;font-weight:700;margin-bottom:6px;">Property Detail</div>
      <b>Property ID:</b> {escape(str(row.get("property_id", "")))}<br>
      <b>Address:</b> {escape(str(row.get("FULLADDR", "")))}<br>
      <b>Latitude:</b> {float(row["parcel_latitude"]):.6f}<br>
      <b>Longitude:</b> {float(row["parcel_longitude"]):.6f}<br>
      <b>Image count:</b> {int(row["image_count"]) if pd.notna(row.get("image_count")) else 0}<br>
      <b>Sale Price:</b> {sale_price_str}<br>
      <b>Last Update:</b> {escape(str(last_update_str))}<br>
      <b>Vacant Indicator:</b> {vacant_str}<br>
      <b>Condition Features:</b> {escape(feature_html)}<br>
      <hr style="margin:8px 0;">
      {''.join(image_blocks) if image_blocks else "<div>No matched images</div>"}
    </div>
    """
    return popup_html


# ------------------------------------------------------------------
# 3) Build JSON records for client-side filtering
# ------------------------------------------------------------------
feature_options = []
for feature in feature_cols:
    feature_options.append({
        "value": feature,
        "label": prettify_feature_name(feature)
    })

price_series = pd.to_numeric(df_parcel_wgs84["property_sale_price"], errors="coerce")
price_non_null = price_series.dropna()

if len(price_non_null) > 0:
    price_min = float(price_non_null.min())
    price_max = float(price_non_null.max())
else:
    price_min = 0.0
    price_max = 1.0

records = []
for _, row in df_parcel_wgs84.iterrows():
    condition_flags = {}
    for feature in feature_cols:
        value = row.get(feature, 0)
        try:
            condition_flags[feature] = int(pd.notna(value) and int(value) == 1)
        except Exception:
            condition_flags[feature] = 0

    sale_price = row.get("property_sale_price", np.nan)
    sale_price_json = None if pd.isna(sale_price) else float(sale_price)

    vacant_value = row.get("vacant_indicator", np.nan)
    if pd.isna(vacant_value):
        status_value = "unknown"
    elif int(vacant_value) == 1:
        status_value = "vacant"
    elif int(vacant_value) == 0:
        status_value = "not_vacant"
    else:
        status_value = "unknown"

    records.append({
        "lat": float(row["parcel_latitude"]),
        "lon": float(row["parcel_longitude"]),
        "property_id": str(row.get("property_id", "")),
        "address": str(row.get("FULLADDR", "")),
        "status": status_value,
        "sale_price": sale_price_json,
        "popup_html": make_property_popup(row),
        "conditions": condition_flags,
    })

records_json = json.dumps(records)
feature_options_json = json.dumps(feature_options)

print("Number of records passed to JS:", len(records))
print("Feature columns:", feature_cols)
print("Sample conditions:", records[0]["conditions"] if len(records) > 0 else "NO RECORDS")


# ------------------------------------------------------------------
# 4) Add custom HTML + JavaScript UI for true filtering
# ------------------------------------------------------------------
map_var = m_features.get_name()

control_html = f"""
<div id="filter-panel" style="
    position: fixed;
    top: 10px;
    left: 50px;
    z-index: 9999;
    width: 370px;
    max-height: 88vh;
    overflow-y: auto;
    background: white;
    border: 2px solid #777;
    border-radius: 8px;
    padding: 12px 12px 10px 12px;
    box-shadow: 0 1px 8px rgba(0,0,0,0.25);
    font-family: sans-serif;
    font-size: 12px;
    line-height: 1.35;
">
  <div style="font-size:14px;font-weight:700;margin-bottom:8px;">Map Filters</div>

  <div style="margin-bottom:10px;">
    <div style="font-weight:700;margin-bottom:4px;">1) Property Status</div>
    <select id="status-filter" style="width:100%;padding:4px;">
      <option value="all" selected>All parcels</option>
      <option value="vacant">Vacant only</option>
      <option value="not_vacant">Not vacant only</option>
    </select>
  </div>

  <div style="margin-bottom:10px;">
    <div style="font-weight:700;margin-bottom:4px;">2) Building Condition</div>

    <div style="margin-bottom:6px;">
      <button id="select-all-conditions" type="button" style="padding:4px 8px;margin-right:6px;">Select all</button>
      <button id="clear-all-conditions" type="button" style="padding:4px 8px;">Clear all</button>
    </div>

    <div id="condition-checkboxes" style="
        border:1px solid #ccc;
        border-radius:6px;
        padding:8px;
        max-height:180px;
        overflow-y:auto;
        background:#fafafa;
    ">
      Loading conditions...
    </div>

    <div style="margin-top:6px;">
      <label style="margin-right:10px;">
        <input type="radio" name="condition-mode" id="condition-mode-any" value="any" checked>
        Match any selected
      </label>
      <label>
        <input type="radio" name="condition-mode" id="condition-mode-all" value="all">
        Match all selected
      </label>
    </div>
  </div>

  <div style="margin-bottom:10px;">
    <div style="font-weight:700;margin-bottom:4px;">3) Sale Price Range</div>

    <label style="display:flex;align-items:center;gap:6px;margin-bottom:6px;">
      <input id="enable-price-filter" type="checkbox" />
      <span>Enable sale price filter</span>
    </label>

    <div style="display:flex;gap:6px;">
      <div style="flex:1;">
        <div style="font-size:11px;color:#555;">Min</div>
        <input id="price-min" type="number" style="width:100%;padding:4px;" />
      </div>
      <div style="flex:1;">
        <div style="font-size:11px;color:#555;">Max</div>
        <input id="price-max" type="number" style="width:100%;padding:4px;" />
      </div>
    </div>

    <div style="margin-top:6px;">
      <button id="reset-price" type="button" style="padding:4px 8px;">Reset price range</button>
    </div>
  </div>

  <div style="margin-bottom:10px;">
    <label style="display:flex;align-items:center;gap:6px;">
      <input id="color-by-price" type="checkbox" />
      <span>Color visible dots by sale price</span>
    </label>
  </div>

  <div style="display:flex;gap:6px;margin-bottom:8px;">
    <button id="apply-filters" type="button" style="padding:5px 10px;">Apply</button>
    <button id="reset-all-filters" type="button" style="padding:5px 10px;">Reset all</button>
  </div>

  <div id="filter-summary" style="font-size:11px;color:#444;background:#f7f7f7;padding:6px;border-radius:6px;">
    Loading filters...
  </div>
</div>

<div id="price-legend" style="
    display:none;
    position: fixed;
    top: 10px;
    right: 20px;
    z-index: 9999;
    width: 210px;
    background: white;
    border: 2px solid #777;
    border-radius: 8px;
    padding: 10px;
    box-shadow: 0 1px 8px rgba(0,0,0,0.25);
    font-family: sans-serif;
    font-size: 12px;
">
  <div style="font-weight:700;margin-bottom:6px;">Property Sale Price</div>
  <div style="height:12px;border-radius:6px;background: linear-gradient(to right, green, yellow, orange, red); margin-bottom:6px;"></div>
  <div style="display:flex;justify-content:space-between;font-size:11px;">
    <span id="legend-min"></span>
    <span id="legend-max"></span>
  </div>
</div>

<script>
document.addEventListener("DOMContentLoaded", function() {{

    function startMapFilters() {{
        const mapObj = window["{map_var}"];
        if (!mapObj) {{
            setTimeout(startMapFilters, 200);
            return;
        }}

        const records = {records_json};
        const featureOptions = {feature_options_json};

        let layerGroup = L.layerGroup().addTo(mapObj);

        const statusSelect = document.getElementById("status-filter");
        const conditionContainer = document.getElementById("condition-checkboxes");
        const enablePriceFilterCheckbox = document.getElementById("enable-price-filter");
        const priceMinInput = document.getElementById("price-min");
        const priceMaxInput = document.getElementById("price-max");
        const colorByPriceCheckbox = document.getElementById("color-by-price");
        const applyBtn = document.getElementById("apply-filters");
        const resetPriceBtn = document.getElementById("reset-price");
        const resetAllBtn = document.getElementById("reset-all-filters");
        const selectAllConditionsBtn = document.getElementById("select-all-conditions");
        const clearAllConditionsBtn = document.getElementById("clear-all-conditions");
        const summaryDiv = document.getElementById("filter-summary");
        const legendDiv = document.getElementById("price-legend");
        const legendMin = document.getElementById("legend-min");
        const legendMax = document.getElementById("legend-max");
        const conditionModeAny = document.getElementById("condition-mode-any");
        const conditionModeAll = document.getElementById("condition-mode-all");

        const DEFAULT_MIN = {price_min};
        const DEFAULT_MAX = {price_max};

        function fmtDollar(x) {{
            if (x === null || x === undefined || isNaN(x)) return "NA";
            return "$" + Number(x).toLocaleString(undefined, {{maximumFractionDigits: 0}});
        }}

        function clamp(value, lo, hi) {{
            return Math.min(Math.max(value, lo), hi);
        }}

        function interpolateColor(value, minVal, maxVal) {{
            if (value === null || value === undefined || isNaN(value)) return "#808080";
            if (maxVal <= minVal) return "orange";

            const t = (value - minVal) / (maxVal - minVal);

            if (t <= 1/3) {{
                const u = t / (1/3);
                const r = Math.round(0 + u * 255);
                const g = 128 + Math.round(u * (255 - 128));
                return `rgb(${{r}},${{g}},0)`;
            }} else if (t <= 2/3) {{
                const u = (t - 1/3) / (1/3);
                const r = 255;
                const g = 255 - Math.round(u * (255 - 165));
                return `rgb(${{r}},${{g}},0)`;
            }} else {{
                const u = (t - 2/3) / (1/3);
                const r = 255;
                const g = 165 - Math.round(u * 165);
                return `rgb(${{r}},${{g}},0)`;
            }}
        }}

        function buildConditionCheckboxes() {{
            conditionContainer.innerHTML = "";

            featureOptions.forEach(opt => {{
                const row = document.createElement("label");
                row.style.display = "flex";
                row.style.alignItems = "center";
                row.style.gap = "6px";
                row.style.marginBottom = "4px";
                row.style.cursor = "pointer";

                const checkbox = document.createElement("input");
                checkbox.type = "checkbox";
                checkbox.className = "condition-checkbox";
                checkbox.value = opt.value;

                const span = document.createElement("span");
                span.textContent = opt.label;

                row.appendChild(checkbox);
                row.appendChild(span);
                conditionContainer.appendChild(row);
            }});
        }}

        function getSelectedConditions() {{
            return Array.from(document.querySelectorAll(".condition-checkbox:checked"))
                .map(cb => cb.value);
        }}

        function getSelectedConditionLabels() {{
            return Array.from(document.querySelectorAll(".condition-checkbox:checked"))
                .map(cb => {{
                    const labelText = cb.parentElement ? cb.parentElement.innerText.trim() : cb.value;
                    return labelText;
                }});
        }}

        function selectAllConditions() {{
            document.querySelectorAll(".condition-checkbox").forEach(cb => {{
                cb.checked = true;
            }});
        }}

        function clearAllConditions() {{
            document.querySelectorAll(".condition-checkbox").forEach(cb => {{
                cb.checked = false;
            }});
            conditionModeAny.checked = true;
        }}

        function resetPriceInputs() {{
            priceMinInput.value = Math.round(DEFAULT_MIN);
            priceMaxInput.value = Math.round(DEFAULT_MAX);
        }}

        function passesStatusFilter(record, selectedStatus) {{
            if (selectedStatus === "all") return true;
            return record.status === selectedStatus;
        }}

        function passesConditionFilter(record, selectedConditions, mode) {{
            if (!selectedConditions || selectedConditions.length === 0) {{
                return true;
            }}

            if (!record.conditions) {{
                return false;
            }}

            if (mode === "all") {{
                return selectedConditions.every(cond => record.conditions[cond] === 1);
            }}

            return selectedConditions.some(cond => record.conditions[cond] === 1);
        }}

        function passesPriceFilter(record, minPrice, maxPrice, enablePriceFilter) {{
            if (!enablePriceFilter) return true;
            if (record.sale_price === null || record.sale_price === undefined || isNaN(record.sale_price)) return false;
            return record.sale_price >= minPrice && record.sale_price <= maxPrice;
        }}

        function updateLegend(showLegend, minVal, maxVal) {{
            if (showLegend) {{
                legendDiv.style.display = "block";
                legendMin.textContent = fmtDollar(minVal);
                legendMax.textContent = fmtDollar(maxVal);
            }} else {{
                legendDiv.style.display = "none";
            }}
        }}

        function renderFilteredMarkers() {{
            const selectedStatus = statusSelect.value;
            const selectedConditions = getSelectedConditions();
            const selectedConditionLabels = getSelectedConditionLabels();
            const conditionMode = document.querySelector('input[name="condition-mode"]:checked').value;
            const enablePriceFilter = enablePriceFilterCheckbox.checked;

            let minPrice = parseFloat(priceMinInput.value);
            let maxPrice = parseFloat(priceMaxInput.value);

            if (isNaN(minPrice)) minPrice = DEFAULT_MIN;
            if (isNaN(maxPrice)) maxPrice = DEFAULT_MAX;

            minPrice = clamp(minPrice, DEFAULT_MIN, DEFAULT_MAX);
            maxPrice = clamp(maxPrice, DEFAULT_MIN, DEFAULT_MAX);

            if (minPrice > maxPrice) {{
                const tmp = minPrice;
                minPrice = maxPrice;
                maxPrice = tmp;
            }}

            priceMinInput.value = Math.round(minPrice);
            priceMaxInput.value = Math.round(maxPrice);

            const filtered = records.filter(r =>
                passesStatusFilter(r, selectedStatus) &&
                passesConditionFilter(r, selectedConditions, conditionMode) &&
                passesPriceFilter(r, minPrice, maxPrice, enablePriceFilter)
            );

            layerGroup.clearLayers();

            const visiblePrices = filtered
                .map(r => r.sale_price)
                .filter(v => v !== null && v !== undefined && !isNaN(v));

            const colorByPrice = colorByPriceCheckbox.checked && visiblePrices.length > 0;
            const colorMin = visiblePrices.length > 0 ? Math.min(...visiblePrices) : minPrice;
            const colorMax = visiblePrices.length > 0 ? Math.max(...visiblePrices) : maxPrice;

            filtered.forEach(r => {{
                let color = "#2E5EAA";

                if (colorByPrice && r.sale_price !== null && r.sale_price !== undefined && !isNaN(r.sale_price)) {{
                    color = interpolateColor(r.sale_price, colorMin, colorMax);
                }}

                const marker = L.circleMarker([r.lat, r.lon], {{
                    radius: 5,
                    color: color,
                    fillColor: color,
                    fillOpacity: 0.9,
                    weight: 1
                }});

                marker.bindPopup(r.popup_html, {{maxWidth: 420}});
                marker.bindTooltip(r.address || r.property_id || "Property");
                marker.addTo(layerGroup);
            }});

            updateLegend(colorByPrice, colorMin, colorMax);

            const statusLabel = statusSelect.options[statusSelect.selectedIndex].text;
            const conditionLabel = selectedConditionLabels.length > 0
                ? selectedConditionLabels.join(", ")
                : "All conditions";

            summaryDiv.innerHTML = `
                <b>Visible parcels:</b> ${{filtered.length}}<br>
                <b>Status:</b> ${{statusLabel}}<br>
                <b>Condition:</b> ${{conditionLabel}}<br>
                <b>Condition mode:</b> ${{conditionMode.toUpperCase()}}<br>
                <b>Sale price filter:</b> ${{enablePriceFilter ? (fmtDollar(minPrice) + " to " + fmtDollar(maxPrice)) : "Off"}}<br>
                <b>Color by price:</b> ${{colorByPrice ? "On" : "Off"}}
            `;
        }}

        buildConditionCheckboxes();
        resetPriceInputs();
        clearAllConditions();
        renderFilteredMarkers();

        applyBtn.addEventListener("click", renderFilteredMarkers);

        resetPriceBtn.addEventListener("click", function() {{
            resetPriceInputs();
            renderFilteredMarkers();
        }});

        resetAllBtn.addEventListener("click", function() {{
            statusSelect.value = "all";
            clearAllConditions();
            enablePriceFilterCheckbox.checked = false;
            colorByPriceCheckbox.checked = false;
            resetPriceInputs();
            renderFilteredMarkers();
        }});

        selectAllConditionsBtn.addEventListener("click", function() {{
            selectAllConditions();
            renderFilteredMarkers();
        }});

        clearAllConditionsBtn.addEventListener("click", function() {{
            clearAllConditions();
            renderFilteredMarkers();
        }});

        statusSelect.addEventListener("change", renderFilteredMarkers);
        enablePriceFilterCheckbox.addEventListener("change", renderFilteredMarkers);
        colorByPriceCheckbox.addEventListener("change", renderFilteredMarkers);
        conditionModeAny.addEventListener("change", renderFilteredMarkers);
        conditionModeAll.addEventListener("change", renderFilteredMarkers);

        conditionContainer.addEventListener("change", function(e) {{
            if (e.target && e.target.classList.contains("condition-checkbox")) {{
                renderFilteredMarkers();
            }}
        }});

        priceMinInput.addEventListener("keydown", function(e) {{
            if (e.key === "Enter") renderFilteredMarkers();
        }});

        priceMaxInput.addEventListener("keydown", function(e) {{
            if (e.key === "Enter") renderFilteredMarkers();
        }});
    }}

    startMapFilters();
}});
</script>
"""

m_features.get_root().html.add_child(Element(control_html))

# ------------------------------------------------------------------
# 5) Save HTML
# ------------------------------------------------------------------
output_html = output_dir / "parcel_feature_map_three_filters_checkbox_conditions.html"
m_features.save(output_html)

print(f"Map saved -> {output_html.resolve()}")
print("Open this HTML file in your browser instead of rendering it inside the notebook.")

Number of records passed to JS: 505
Feature columns: ['Crooked_Roofline', 'Broken_Steps', 'Missing_Bricks', 'Peeling_Paint', 'Boarded_Windows', 'Boarded_Doors', 'Plant_Overgrowth', 'Fire_Damage', 'Cracked_Sidewalk', 'Green_Space']
Sample conditions: {'Crooked_Roofline': 0, 'Broken_Steps': 0, 'Missing_Bricks': 0, 'Peeling_Paint': 0, 'Boarded_Windows': 0, 'Boarded_Doors': 0, 'Plant_Overgrowth': 0, 'Fire_Damage': 0, 'Cracked_Sidewalk': 0, 'Green_Space': 0}
Map saved -> /Users/jiaweihu/Desktop/BBxB/Spatial Join/outputs/parcel_feature_map_three_filters_checkbox_conditions.html
Open this HTML file in your browser instead of rendering it inside the notebook.


In [16]:
# --- Diagnostics: verify all best-matched images are represented in parcel_matches ---

parcel_match_imgs = set()
for s in parcel_matches["image_names"].dropna():
    for img in str(s).split("|"):
        parcel_match_imgs.add(img.strip())

missing_imgs = set(best_match["filename"]) - parcel_match_imgs
extra_imgs = parcel_match_imgs - set(best_match["filename"])

print("Images in best_match but not represented in parcel_matches:", missing_imgs)
print("Images represented in parcel_matches but not in best_match:", extra_imgs)
print("Count in best_match:", best_match["filename"].nunique())
print("Count represented through parcel_matches:", len(parcel_match_imgs))

Images in best_match but not represented in parcel_matches: set()
Images represented in parcel_matches but not in best_match: set()
Count in best_match: 578
Count represented through parcel_matches: 578


In [17]:
# --- Export outputs ---

# ========= 1) image-level results =========
best_match_export = best_match.copy()

# 把额外 geometry 列转成 WKT，避免 to_file 报错
extra_geom_cols_best = ["camera_point", "parcel_geom", "parcel_centroid_proj", "parcel_rep_pt_proj"]
for col in extra_geom_cols_best:
    if col in best_match_export.columns:
        best_match_export[f"{col}_wkt"] = best_match_export[col].to_wkt()

# GeoJSON 只保留一个 active geometry 列
best_match_geojson = best_match_export.drop(
    columns=[c for c in extra_geom_cols_best if c in best_match_export.columns],
    errors="ignore"
).copy()

best_match_geojson = gpd.GeoDataFrame(
    best_match_geojson,
    geometry="geometry",
    crs=best_match.crs
).to_crs("EPSG:4326")

best_match_geojson.to_file("best_match.geojson", driver="GeoJSON")

# CSV：把 active geometry 也转成 WKT
best_match_csv = best_match_export.copy()
best_match_csv["geometry_wkt"] = best_match_csv.geometry.to_wkt()
best_match_csv = pd.DataFrame(best_match_csv.drop(columns="geometry"))
best_match_csv.to_csv("best_match.csv", index=False)

# ========= 2) parcel-level matched results =========
parcel_matches_export = parcel_matches.copy()

extra_geom_cols_parcel = ["parcel_geom", "parcel_centroid_proj", "parcel_rep_pt_proj"]
for col in extra_geom_cols_parcel:
    if col in parcel_matches_export.columns:
        parcel_matches_export[f"{col}_wkt"] = parcel_matches_export[col].to_wkt()

parcel_matches_geojson = parcel_matches_export.drop(
    columns=[c for c in extra_geom_cols_parcel if c in parcel_matches_export.columns],
    errors="ignore"
).copy()

parcel_matches_geojson = gpd.GeoDataFrame(
    parcel_matches_geojson,
    geometry="geometry",
    crs=parcel_matches.crs
).to_crs("EPSG:4326")

parcel_matches_geojson.to_file("parcel_matches.geojson", driver="GeoJSON")

parcel_matches_csv = parcel_matches_export.copy()
parcel_matches_csv["geometry_wkt"] = parcel_matches_csv.geometry.to_wkt()
parcel_matches_csv = pd.DataFrame(parcel_matches_csv.drop(columns="geometry"))
parcel_matches_csv.to_csv("parcel_matches.csv", index=False)

print("Saved -> best_match.geojson")
print("Saved -> best_match.csv")
print("Saved -> parcel_matches.geojson")
print("Saved -> parcel_matches.csv")

Saved -> best_match.geojson
Saved -> best_match.csv
Saved -> parcel_matches.geojson
Saved -> parcel_matches.csv


In [18]:
# best_match_export = best_match.drop(
#     columns=["camera_point", "geometry", "parcel_geom", "parcel_centroid_proj", "parcel_rep_pt_proj", "index_right"],
#     errors="ignore",
# ).copy()
# best_match_export.to_csv(BEST_MATCH_CSV, index=False)

# parcel_matches_for_csv = parcel_matches.drop(
#     columns=["geometry", "parcel_geom", "parcel_centroid_proj", "parcel_rep_pt_proj"],
#     errors="ignore",
# ).copy()
# parcel_matches_for_csv.to_csv(PARCEL_MATCH_CSV, index=False)

# parcel_matches_for_geojson = parcel_matches.drop(
#     columns=["parcel_geom", "parcel_centroid_proj", "parcel_rep_pt_proj"],
#     errors="ignore",
# ).copy()
# parcel_matches_for_geojson.to_file(PARCEL_MATCH_GEOJSON, driver="GeoJSON")

# print("Saved ->", BEST_MATCH_CSV)
# print("Saved ->", PARCEL_MATCH_CSV)
# print("Saved ->", PARCEL_MATCH_GEOJSON)

In [19]:
images_with_candidates = set(candidates_all.loc[candidates_all["property_id"].notna(), "filename"].unique())
images_after_filter = set(candidates["filename"].unique())
best_match_images = set(best_match["filename"].unique())

dropped_by_distance = sorted(images_with_candidates - images_after_filter)
missing_after_best_match = sorted(images_after_filter - best_match_images)

parcel_matched_rows = parcel_matches.loc[parcel_matches["image_count"].notna()].copy()

images_shown_on_parcel_layer = set()
for s in parcel_matched_rows["image_names"].dropna():
    for img in str(s).split("|"):
        img = img.strip()
        if img:
            images_shown_on_parcel_layer.add(img)

not_in_parcel_aggregation = sorted(best_match_images - images_shown_on_parcel_layer)

print("Images dropped by distance filter:", len(dropped_by_distance))
print(dropped_by_distance)
print()
print("Images missing after best_match step:", len(missing_after_best_match))
print(missing_after_best_match)
print()
print("Best-match images not represented in parcel aggregation:", len(not_in_parcel_aggregation))
print(not_in_parcel_aggregation)
print()
print("best_match image count:", len(best_match_images))
print("parcel marker count   :", parcel_matched_rows.shape[0])

Images dropped by distance filter: 0
[]

Images missing after best_match step: 0
[]

Best-match images not represented in parcel aggregation: 0
[]

best_match image count: 578
parcel marker count   : 505


In [16]:
# Secondary parcel-only map removed so the notebook outputs only one Folium map.\nprint("Using only the single filterable Folium map above.")

Map saved -> parcel_map.html (505 parcel markers)


In [31]:
print(best_match["image_path"].head(10).tolist())

['/Users/jiaweihu/Desktop/BBxB/Testing/Baltimore_dataset/testing_image/Run 10_Camera 4 360_115_1.jpg', '/Users/jiaweihu/Desktop/BBxB/Testing/Baltimore_dataset/testing_image/Run 10_Camera 4 360_192_3.jpg', '/Users/jiaweihu/Desktop/BBxB/Testing/Baltimore_dataset/testing_image/Run 10_Camera 4 360_221_1.jpg', '/Users/jiaweihu/Desktop/BBxB/Testing/Baltimore_dataset/testing_image/Run 10_Camera 4 360_238_3.jpg', '/Users/jiaweihu/Desktop/BBxB/Testing/Baltimore_dataset/testing_image/Run 10_Camera 4 360_23_1.jpg', '/Users/jiaweihu/Desktop/BBxB/Testing/Baltimore_dataset/testing_image/Run 10_Camera 4 360_265_1.jpg', '/Users/jiaweihu/Desktop/BBxB/Testing/Baltimore_dataset/testing_image/Run 10_Camera 4 360_272_1.jpg', '/Users/jiaweihu/Desktop/BBxB/Testing/Baltimore_dataset/testing_image/Run 10_Camera 4 360_288_3.jpg', '/Users/jiaweihu/Desktop/BBxB/Testing/Baltimore_dataset/testing_image/Run 10_Camera 4 360_36_1.jpg', '/Users/jiaweihu/Desktop/BBxB/Testing/Baltimore_dataset/testing_image/Run 10_Camera